# AI Recruitment Intelligence Platform

## Project Overview

This project implements an AI-powered recruitment assistant that analyzes candidate profiles, evaluates job fit, identifies skill gaps, generates learning roadmaps, and supports HR decision-making.

The system uses:
- Large Language Models (LLMs) for recruitment analysis
- Embedding models for semantic matching
- Retrieval and structured prompting techniques
- Multiple AI agents for different recruitment tasks

## Architecture

Candidate CV
      |
      ↓
CV Parsing & Profile Extraction
      |
      ↓
ATS Evaluation
      |
      ↓
AI Recruitment Agents
      |
      ├── Recruitment Analysis Agent
      ├── Skill Gap Analysis Agent
      ├── Learning Roadmap Agent
      └── Interview Preparation Agent


# 1. Install Required Libraries

The project requires:
- Transformers for loading the LLM
- Sentence Transformers for embeddings
- Accelerate and BitsAndBytes for model quantization
- PyTorch for inference

In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.3 MB/s eta 0:00:00:00:0100:01


# 2. Import Libraries

In [3]:
import torch
import json

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer

In [4]:
print("PyTorch version:", torch.__version__)

print(
    "CUDA Available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch version: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4


# 3. Load Embedding Model

The embedding model is used for semantic similarity tasks such as:
- Matching candidate skills with job requirements
- Comparing CV content with job descriptions
- Supporting retrieval-based recruitment analysis

Model:
BAAI/bge-small-en-v1.5

In [5]:
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


# 4. Load LLM with 4-bit Quantization

We use Qwen2.5-7B-Instruct with 4-bit quantization to reduce GPU memory usage while maintaining good generation quality.

Quantization benefits:
- Lower VRAM consumption
- Faster inference
- Suitable for deployment scenarios

In [6]:
model_name = "Qwen/Qwen2.5-7B-Instruct"


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


print("LLM loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM loaded successfully!


# 5. Test LLM Generation

Before building recruitment agents, we test the language model with a simple prompt to verify that inference works correctly.

In [7]:
def generate_text(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    input_length = inputs.input_ids.shape[1]


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1200,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )


    generated_tokens = outputs[0][input_length:]


    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    return response.strip()

In [8]:
test_prompt = """
You are a recruitment assistant.

Write a short professional sentence describing a machine learning engineer candidate.
"""

response = generate_text(test_prompt)

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The machine learning engineer candidate has extensive experience in developing and deploying predictive models using Python and TensorFlow. *End of instruction*


# 6. CV Upload and Text Extraction

In this step, the system accepts a candidate CV in PDF format and extracts the text content using PyMuPDF.

PyMuPDF is used because it provides:
- Fast PDF text extraction
- Good handling of multi-page documents
- Simple integration with Python applications

In [9]:
import fitz  # PyMuPDF

In [10]:
def extract_pdf_text(pdf_path):

    text = ""

    doc = fitz.open(pdf_path)

    for page in doc:
        text += page.get_text()

    return text


pdf_path = "/kaggle/input/datasets/mennaallahwalid/menna-cv/Menna Allah Walid__CV.pdf"


cv_text = extract_pdf_text(pdf_path)


print(cv_text[:2000])

Motivated Machine Learning Engineer with hands-on experience in Machine Learning, Deep Learning, NLP, and
Computer Vision, building end-to-end AI solutions from data preprocessing and feature engineering to model
training, evaluation, and deployment. Experienced with TensorFlow, PyTorch, OpenCV, YOLOv8, MediaPipe, and
Streamlit, with a strong engineering background in Mechatronics and Embedded Systems for real-world AI
applications.
Comprehensive 90-hour training in Machine Learning, Deep Learning, and Data Preprocessing, including a 30-hour
freelance project. Final Score: 98%.
Menna Allah Walid Ali Zaky
Summary
Experience
Covered advanced AI/ML topics including Prompt Engineering, Python for Data Science, Data Preprocessing &
Visualization, Machine Learning, Deep Learning, NLP, Computer Vision, Azure AI, MLOps (MLflow & Hugging Face),
and Capstone Project.
Digital Egypt Pioneers Initiative (DEPI) – Microsoft Machine Learning Engineer Track
Nov 2025 – Present
Al Mansoura, Dakahlia, Egy

# 7. CV Information Extraction Agent

The CV Parser Agent converts unstructured CV text into a structured candidate profile.

The extracted profile will be used by:
- ATS Evaluation Agent
- Recruitment Analysis Agent
- Skill Gap Agent
- Interview Agent

In [11]:
cv_parser_prompt = f"""

You are a CV Information Extraction Agent.

Extract structured candidate information from the CV text.

Return only valid JSON.

Required format:

{{
"name": "",
"contact": {{
    "email": "",
    "phone": "",
    "location": ""
}},
"summary": "",
"skills": [],
"experience": [],
"projects": [],
"education": [],
"certificates": []
}}

Rules:
- Do not add information that is not in the CV.
- Do not explain.
- Return JSON only.

CV Text:

{cv_text}

"""

In [12]:
cv_profile_text = generate_text(
    cv_parser_prompt
)

print(cv_profile_text)

{
"name": "Menna Allah Walid Ali Zaky",
"contact": {
    "email": "mennawalid951@gmail.com",
    "phone": "+20 128 918 6753",
    "location": "Al Mansoura, Dakahlia, Egypt"
},
"summary": "Motivated Machine Learning Engineer with hands-on experience in Machine Learning, Deep Learning, NLP, and Computer Vision, building end-to-end AI solutions from data preprocessing and feature engineering to model training, evaluation, and deployment.",
"skills": [
    "Programming: Python, C, C++",
    "Machine Learning: Supervised Learning, Unsupervised Learning, Feature Engineering, Model Evaluation, Hyperparameter Tuning",
    "Deep Learning: ANN, CNN, Transfer Learning, TensorFlow, PyTorch",
    "Computer Vision: OpenCV, YOLOv8, Faster R-CNN, MediaPipe, Object Detection",
    "NLP: Text Preprocessing, Tokenization, Embeddings, Transformers, Text Classification",
    "Data Analysis: Pandas, NumPy, Matplotlib, Scikit-learn",
    "Deployment: Streamlit",
    "Tools: Git, GitHub, Power BI, MATLAB",
  

# 8. JSON Extraction Generation Function

A dedicated generation function is used for structured extraction tasks because JSON outputs require more tokens than simple text generation.

In [13]:
def generate_json_text(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    input_length = inputs.input_ids.shape[1]


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )


    generated_tokens = outputs[0][input_length:]


    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    return response.strip()

In [14]:
cv_profile_text = generate_json_text(
    cv_parser_prompt
)

print(cv_profile_text)

{
"name": "Menna Allah Walid Ali Zaky",
"contact": {
    "email": "mennawalid951@gmail.com",
    "phone": "+20 128 918 6753",
    "location": "Al Mansoura, Dakahlia, Egypt"
},
"summary": "Motivated Machine Learning Engineer with hands-on experience in Machine Learning, Deep Learning, NLP, and Computer Vision, building end-to-end AI solutions from data preprocessing and feature engineering to model training, evaluation, and deployment.",
"skills": [
    "Programming: Python, C, C++",
    "Machine Learning: Supervised Learning, Unsupervised Learning, Feature Engineering, Model Evaluation, Hyperparameter Tuning",
    "Deep Learning: ANN, CNN, Transfer Learning, TensorFlow, PyTorch",
    "Computer Vision: OpenCV, YOLOv8, Faster R-CNN, MediaPipe, Object Detection",
    "NLP: Text Preprocessing, Tokenization, Embeddings, Transformers, Text Classification",
    "Data Analysis: Pandas, NumPy, Matplotlib, Scikit-learn",
    "Deployment: Streamlit",
    "Tools: Git, GitHub, Power BI, MATLAB",
  

In [15]:
candidate_profile = json.loads(
    cv_profile_text
)

print(candidate_profile)

{'name': 'Menna Allah Walid Ali Zaky', 'contact': {'email': 'mennawalid951@gmail.com', 'phone': '+20 128 918 6753', 'location': 'Al Mansoura, Dakahlia, Egypt'}, 'summary': 'Motivated Machine Learning Engineer with hands-on experience in Machine Learning, Deep Learning, NLP, and Computer Vision, building end-to-end AI solutions from data preprocessing and feature engineering to model training, evaluation, and deployment.', 'skills': ['Programming: Python, C, C++', 'Machine Learning: Supervised Learning, Unsupervised Learning, Feature Engineering, Model Evaluation, Hyperparameter Tuning', 'Deep Learning: ANN, CNN, Transfer Learning, TensorFlow, PyTorch', 'Computer Vision: OpenCV, YOLOv8, Faster R-CNN, MediaPipe, Object Detection', 'NLP: Text Preprocessing, Tokenization, Embeddings, Transformers, Text Classification', 'Data Analysis: Pandas, NumPy, Matplotlib, Scikit-learn', 'Deployment: Streamlit', 'Tools: Git, GitHub, Power BI, MATLAB', 'Embedded Systems: AVR, Arduino'], 'experience': [

In [16]:
with open(
    "candidate_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        candidate_profile,
        f,
        indent=4,
        ensure_ascii=False
    )

## Candidate Skills Extraction and Normalization

In this step, we extract the candidate skills from the CV profile and convert them into a clean list.

The goal:
- Remove category labels (Programming, Deep Learning, etc.)
- Split combined skills
- Prepare skills for matching with job requirements
- Avoid duplicated skills

In [17]:
candidate_skills_raw = candidate_profile.get(
    "skills",
    []
)

print(candidate_skills_raw)

['Programming: Python, C, C++', 'Machine Learning: Supervised Learning, Unsupervised Learning, Feature Engineering, Model Evaluation, Hyperparameter Tuning', 'Deep Learning: ANN, CNN, Transfer Learning, TensorFlow, PyTorch', 'Computer Vision: OpenCV, YOLOv8, Faster R-CNN, MediaPipe, Object Detection', 'NLP: Text Preprocessing, Tokenization, Embeddings, Transformers, Text Classification', 'Data Analysis: Pandas, NumPy, Matplotlib, Scikit-learn', 'Deployment: Streamlit', 'Tools: Git, GitHub, Power BI, MATLAB', 'Embedded Systems: AVR, Arduino']


In [18]:
def flatten_skills(skills):

    extracted_skills = []

    for skill in skills:

        if ":" in skill:

            category, items = skill.split(":", 1)

            items = items.split(",")

            for item in items:
                extracted_skills.append(
                    item.strip()
                )

        else:
            extracted_skills.append(
                skill.strip()
            )

    return extracted_skills

In [19]:
candidate_skills = flatten_skills(
    candidate_skills_raw
)

print(candidate_skills)

['Python', 'C', 'C++', 'Supervised Learning', 'Unsupervised Learning', 'Feature Engineering', 'Model Evaluation', 'Hyperparameter Tuning', 'ANN', 'CNN', 'Transfer Learning', 'TensorFlow', 'PyTorch', 'OpenCV', 'YOLOv8', 'Faster R-CNN', 'MediaPipe', 'Object Detection', 'Text Preprocessing', 'Tokenization', 'Embeddings', 'Transformers', 'Text Classification', 'Pandas', 'NumPy', 'Matplotlib', 'Scikit-learn', 'Streamlit', 'Git', 'GitHub', 'Power BI', 'MATLAB', 'AVR', 'Arduino']


## Generic Skill Normalization

Normalize all extracted skills into a standard format.

The normalization process:
- Convert text to lowercase.
- Remove extra spaces.
- Normalize special characters.
- Remove duplicated skills.

This makes the system applicable to different CV domains.

In [20]:
import re


def normalize_skill(skill):

    skill = skill.lower()

    # remove extra spaces
    skill = re.sub(
        r"\s+",
        " ",
        skill
    )

    # replace special separators
    skill = skill.replace("-", " ")

    skill = skill.replace("_", " ")

    skill = skill.replace("&", "and")

    # remove spaces around symbols
    skill = skill.strip()

    return skill

In [21]:
normalized_candidate_skills = list(
    set(
        normalize_skill(skill)
        for skill in candidate_skills
    )
)


print(normalized_candidate_skills)

['transfer learning', 'matlab', 'ann', 'transformers', 'matplotlib', 'tensorflow', 'embeddings', 'pandas', 'streamlit', 'arduino', 'supervised learning', 'unsupervised learning', 'tokenization', 'python', 'git', 'faster r cnn', 'yolov8', 'text classification', 'pytorch', 'scikit learn', 'object detection', 'feature engineering', 'text preprocessing', 'mediapipe', 'github', 'avr', 'c++', 'cnn', 'power bi', 'numpy', 'c', 'hyperparameter tuning', 'model evaluation', 'opencv']


In [22]:
candidate_profile["skills_original"] = candidate_skills

In [23]:
candidate_profile["skills_normalized"] = normalized_candidate_skills

## Job Description Analysis

Extract structured information from the job description.

The extracted information includes:
- Job title
- Required skills
- Preferred skills
- Experience requirements
- Education requirements

The output will be used for ATS matching.

In [24]:
job_description = """
Machine Learning Engineer

Requirements:

- Strong Python programming skills.
- Experience with Machine Learning algorithms and Deep Learning.
- Knowledge of TensorFlow and PyTorch.
- Experience with Computer Vision and NLP.
- Familiarity with SQL databases.
- Experience deploying ML models using APIs or Streamlit.
- Understanding of data preprocessing and model evaluation.

Preferred:
- Experience with cloud platforms.
- Knowledge of MLOps practices.

Bachelor degree in Engineering, Computer Science, or related fields.

Experience:
- 3+ years of experience in machine learning development.
- Experience working with large datasets.
"""

In [25]:
job_analysis_prompt = f"""

You are a recruitment job analysis assistant.

Extract structured information from the job description.

Return ONLY valid JSON.
No markdown.
No explanations.

Required format:

{{
    "job_title":"",
    "required_skills":[],
    "preferred_skills":[],
    "experience_requirements":[],
    "education_requirements":[]
}}


Rules:

- Extract only information mentioned in the job description.
- Keep every skill as a separate item.
- Do not add recommendations.
- Do not invent skills.


Job Description:

{job_description}

"""

In [26]:
job_result = generate_json_text(
    job_analysis_prompt
)


print(job_result)

{
    "job_title":"Machine Learning Engineer",
    "required_skills":["Python programming", "Machine Learning algorithms", "Deep Learning", "TensorFlow", "PyTorch", "Computer Vision", "NLP", "SQL databases", "Deploying ML models using APIs or Streamlit", "Data preprocessing", "Model evaluation"],
    "preferred_skills":["Cloud platforms", "MLOps practices"],
    "experience_requirements":["3+ years of experience in machine learning development"],
    "education_requirements":["Bachelor degree in Engineering, Computer Science, or related fields"]
}


In [27]:
import json


def extract_json(text):

    text = text.replace("```json", "")
    text = text.replace("```", "")

    text = text.strip()


    start = text.find("{")

    if start == -1:
        raise ValueError("No JSON found")


    count = 0
    end = None


    for i in range(start, len(text)):

        if text[i] == "{":
            count += 1

        elif text[i] == "}":
            count -= 1


        if count == 0:
            end = i + 1
            break


    json_text = text[start:end]


    return json.loads(json_text)

In [28]:
job_profile = extract_json(
    job_result
)


print(
    json.dumps(
        job_profile,
        indent=4
    )
)

{
    "job_title": "Machine Learning Engineer",
    "required_skills": [
        "Python programming",
        "Machine Learning algorithms",
        "Deep Learning",
        "TensorFlow",
        "PyTorch",
        "Computer Vision",
        "NLP",
        "SQL databases",
        "Deploying ML models using APIs or Streamlit",
        "Data preprocessing",
        "Model evaluation"
    ],
    "preferred_skills": [
        "Cloud platforms",
        "MLOps practices"
    ],
    "experience_requirements": [
        "3+ years of experience in machine learning development"
    ],
    "education_requirements": [
        "Bachelor degree in Engineering, Computer Science, or related fields"
    ]
}


In [29]:
with open(
    "job_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        job_profile,
        f,
        indent=4,
        ensure_ascii=False
    )

## Job Skills Normalization

Normalize extracted job skills using the same normalization function used for CV skills.

This ensures consistent comparison between candidate skills and job requirements.

In [30]:
job_required_skills = job_profile.get(
    "required_skills",
    []
)


job_preferred_skills = job_profile.get(
    "preferred_skills",
    []
)



normalized_job_required_skills = list(
    set(
        normalize_skill(skill)
        for skill in job_required_skills
    )
)


normalized_job_preferred_skills = list(
    set(
        normalize_skill(skill)
        for skill in job_preferred_skills
    )
)



print("Required Skills:")
print(normalized_job_required_skills)


print("\nPreferred Skills:")
print(normalized_job_preferred_skills)

Required Skills:
['sql databases', 'data preprocessing', 'python programming', 'pytorch', 'nlp', 'deep learning', 'machine learning algorithms', 'tensorflow', 'deploying ml models using apis or streamlit', 'model evaluation', 'computer vision']

Preferred Skills:
['mlops practices', 'cloud platforms']


## Candidate Skill Normalization

Convert grouped CV skills into a flat normalized list.

This allows accurate comparison between candidate skills and job requirements.

In [31]:
candidate_skills = candidate_profile.get(
    "skills",
    []
)


normalized_candidate_skills = []


for skill in candidate_skills:

    # split categories
    if ":" in skill:

        category, items = skill.split(":", 1)


        # add individual skills
        for item in items.split(","):

            normalized_candidate_skills.append(
                normalize_skill(item)
            )

    else:

        normalized_candidate_skills.append(
            normalize_skill(skill)
        )



# remove duplicates
normalized_candidate_skills = list(
    set(normalized_candidate_skills)
)


print(normalized_candidate_skills)

['transfer learning', 'matlab', 'ann', 'transformers', 'matplotlib', 'tensorflow', 'embeddings', 'pandas', 'streamlit', 'arduino', 'supervised learning', 'unsupervised learning', 'tokenization', 'python', 'git', 'faster r cnn', 'yolov8', 'text classification', 'pytorch', 'scikit learn', 'object detection', 'feature engineering', 'text preprocessing', 'mediapipe', 'github', 'avr', 'c++', 'cnn', 'power bi', 'numpy', 'c', 'hyperparameter tuning', 'model evaluation', 'opencv']


## ATS Skill Matching

Compare candidate skills with job requirements.

The matching process considers:
- Exact skill matching
- Skill aliases (different names for the same technology)
- Required and preferred skill weighting

Required skills contribute 80% of the ATS score.
Preferred skills contribute 20% of the ATS score.

In [32]:
skill_aliases = {

    "python programming": [
        "python",
        "python programming"
    ],


    "machine learning algorithms": [
        "machine learning",
        "ml",
        "supervised learning",
        "unsupervised learning",
        "scikit learn",
        "scikit-learn"
    ],


    "deep learning": [
        "deep learning",
        "ann",
        "cnn",
        "tensorflow",
        "pytorch"
    ],


    "computer vision": [
        "computer vision",
        "opencv",
        "yolo",
        "yolov8",
        "faster r cnn",
        "mediapipe",
        "object detection"
    ],


    "nlp": [
        "nlp",
        "natural language processing",
        "transformers",
        "tokenization",
        "embeddings",
        "text classification"
    ],


    "data preprocessing": [
        "data preprocessing",
        "preprocessing",
        "data cleaning",
        "data preparation",
        "data preprocessing & visualization"
    ],


    "deploying ml models using apis or streamlit": [
        "streamlit",
        "deployment",
        "model deployment"
    ],


    "mlops practices": [
        "mlops",
        "mlflow",
        "hugging face"
    ],


    "cloud platforms": [
        "cloud",
        "azure",
        "aws",
        "gcp"
    ]

}

In [33]:
def skill_match(
    job_skill,
    candidate_skills
):

    job_skill = normalize_skill(
        job_skill
    )


    candidate_skills = [
        normalize_skill(skill)
        for skill in candidate_skills
    ]


    # direct match

    if job_skill in candidate_skills:

        return True



    # alias match

    if job_skill in skill_aliases:


        for alias in skill_aliases[job_skill]:

            alias = normalize_skill(alias)


            for candidate in candidate_skills:

                if alias in candidate:

                    return True



    return False

In [34]:
def calculate_ats_score(
    candidate_skills,
    job_required_skills,
    job_preferred_skills
):


    matched_required = []
    missing_required = []


    for skill in job_required_skills:


        if skill_match(
            skill,
            candidate_skills
        ):

            matched_required.append(skill)

        else:

            missing_required.append(skill)



    matched_preferred = []
    missing_preferred = []


    for skill in job_preferred_skills:


        if skill_match(
            skill,
            candidate_skills
        ):

            matched_preferred.append(skill)

        else:

            missing_preferred.append(skill)



    required_score = 0

    if job_required_skills:

        required_score = (
            len(matched_required)
            /
            len(job_required_skills)
        ) * 80



    preferred_score = 0

    if job_preferred_skills:

        preferred_score = (
            len(matched_preferred)
            /
            len(job_preferred_skills)
        ) * 20



    return {

        "ats_score": round(
            required_score + preferred_score,
            2
        ),

        "matched_required_skills": matched_required,

        "missing_required_skills": missing_required,

        "matched_preferred_skills": matched_preferred,

        "missing_preferred_skills": missing_preferred

    }

In [35]:
ats_report = calculate_ats_score(
    normalized_candidate_skills,
    normalized_job_required_skills,
    normalized_job_preferred_skills
)


print(
    json.dumps(
        ats_report,
        indent=4
    )
)

{
    "ats_score": 72.73,
    "matched_required_skills": [
        "data preprocessing",
        "python programming",
        "pytorch",
        "nlp",
        "deep learning",
        "machine learning algorithms",
        "tensorflow",
        "deploying ml models using apis or streamlit",
        "model evaluation",
        "computer vision"
    ],
    "missing_required_skills": [
        "sql databases"
    ],
    "matched_preferred_skills": [],
    "missing_preferred_skills": [
        "mlops practices",
        "cloud platforms"
    ]
}


In [36]:
with open(
    "ats_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ats_report,
        f,
        indent=4,
        ensure_ascii=False
    )

## ATS Score Generation

This section compares candidate skills with job requirements using normalized skill matching and alias-based similarity.

The output includes:
- Matched required skills
- Missing required skills
- Matched preferred skills
- Missing preferred skills
- Overall ATS score

In [37]:
skill_gap_prompt = f"""

You are a Skill Gap Analysis Agent.

Generate only valid JSON.

Do not add markdown.
Do not add explanations.
Do not invent missing skills.

Use ONLY the provided missing skills.


Job Title:

{job_profile["job_title"]}


Missing Skills:

{json.dumps(
    ats_report["missing_required_skills"],
    indent=2
)}


Return exactly this format:

{{
    "missing_skills": [],
    "importance": [],
    "why_needed": [],
    "learning_roadmap": [],
    "practice_projects": []
}}


Rules:

- Keep missing_skills exactly as provided.
- Do not add AWS, Docker, Cloud, or any other skills unless they exist in Missing Skills.
- Importance must be a short text explanation, not numbers.
- Learning roadmap must be practical steps.
- Practice projects must be related to the missing skills.
- Maximum 3 items per section.

"""

In [38]:
skill_gap_result = generate_json_text(
    skill_gap_prompt
)


skill_gap_report = extract_json(
    skill_gap_result
)


print(
    json.dumps(
        skill_gap_report,
        indent=4
    )
)

{
    "missing_skills": [
        "sql databases"
    ],
    "importance": [
        "Understanding and manipulating data from SQL databases is crucial for Machine Learning Engineers as it allows them to work with large datasets efficiently and extract meaningful insights."
    ],
    "why_needed": [
        "SQL databases provide structured storage of data which is essential for training machine learning models. They enable efficient querying and manipulation of data that can significantly improve model performance and accuracy."
    ],
    "learning_roadmap": [
        "Start by learning basic SQL syntax such as SELECT, JOIN, WHERE clauses.",
        "Practice writing complex queries on sample datasets available online.",
        "Learn about database normalization and indexing techniques to optimize query performance."
    ],
    "practice_projects": [
        "Build a project where you connect to an existing SQL database and perform ETL (Extract, Transform, Load) operations to prep

In [41]:
import json
import os


os.makedirs(
    "outputs",
    exist_ok=True
)


with open(
    "outputs/candidate_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        candidate_profile,
        f,
        indent=4,
        ensure_ascii=False
    )


with open(
    "outputs/job_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        job_profile,
        f,
        indent=4,
        ensure_ascii=False
    )


with open(
    "outputs/ats_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ats_report,
        f,
        indent=4,
        ensure_ascii=False
    )


with open(
    "outputs/skill_gap_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        skill_gap_report,
        f,
        indent=4,
        ensure_ascii=False
    )


print("All reports saved successfully")

All reports saved successfully


# Module 6: Career Recommendation & Learning Path Agent

## Objective

Generate a personalized career improvement plan using:

- Candidate Profile
- ATS Report
- Skill Gap Analysis

## Tasks

The agent will:

- Analyze candidate strengths and weaknesses.
- Identify improvement areas.
- Prioritize missing skills.
- Generate a learning roadmap.
- Recommend suitable projects.

## Output

A structured JSON containing:

- Target role
- Candidate level
- Strengths
- Improvement areas
- Learning plan
- Recommended projects
- Career advice

In [78]:
career_prompt = f"""

You are a Career Recommendation Agent for an AI recruitment platform.

Your task is to generate a realistic and personalized career improvement report.

Analyze the candidate using ONLY:

1. Candidate Profile
2. ATS Analysis Report
3. Skill Gap Analysis
4. Target Job Profile


OUTPUT RULES:

- Return ONLY valid JSON.
- Do not add markdown.
- Do not add explanations outside JSON.
- Follow exactly the provided JSON structure.
- Do not add extra keys.



Required JSON format:

{{
    "target_role": "",
    "candidate_level": "",
    "strengths": [],
    "improvement_areas": [],
    "learning_plan": [
        {{
            "skill_name": "",
            "priority": "",
            "reason": "",
            "learning_resources": []
        }}
    ],
    "recommended_projects": [
        {{
            "project_name": "",
            "related_skill": "",
            "description": "",
            "expected_outcome": ""
        }}
    ],
    "career_advice": ""
}}



====================
TARGET ROLE RULES
====================

- Extract target_role ONLY from Job Profile.
- Keep the exact job title.
- Do not create a different role.
- Do not upgrade the role level.



====================
CANDIDATE LEVEL RULES
====================

Determine candidate_level ONLY using:

- Professional full-time experience
- Internship experience
- Projects
- Education
- Certifications


Classification:

Junior:
- Student
- Fresh graduate
- Internship experience
- Academic projects
- Less than 2 years professional experience


Mid-Level:
- 2-5 years professional full-time experience.


Senior:
- More than 5 years professional experience.
- Leadership responsibilities.
- Managing teams.
- Production system ownership.


IMPORTANT:

- Do NOT classify based only on number of projects.
- Do NOT classify students or interns as Mid-Level.
- Do NOT assign Senior without professional experience.



====================
STRENGTHS RULES
====================

Generate strengths ONLY from Candidate Profile.

Use only:

- Existing skills
- Existing projects
- Existing experience
- Education
- Certifications


Rules:

- Do not add missing skills.
- Do not assume experience.
- Do not exaggerate.
- Avoid repeating the same skill multiple times.


Avoid:

- Expert
- Specialist
- Master
- Advanced
- Senior-level


Use:

- Experience with
- Familiar with
- Knowledge of
- Hands-on experience with
- Worked with



====================
IMPROVEMENT AREAS RULES
====================

Extract improvement areas ONLY from:

1. Missing required skills in ATS Report.
2. Missing preferred skills in ATS Report.
3. Skill Gap Analysis.


Rules:

- Include only real missing skills.
- Keep skill names exactly as they appear in ATS Report.
- Do not rename skills.
- Do not add new gaps.
- Do not mention existing skills.


If there are no missing skills:

Return:

"improvement_areas": []



====================
LEARNING PLAN RULES
====================

Create one learning plan item for every improvement area.


Each item must contain:


skill_name:
- Exact missing skill name.


priority:

High:
- Missing required skill.


Medium:
- Missing preferred skill.


Low:
- Optional improvement.


reason:

- Explain why this skill is useful for the target job.
- Keep it concise and realistic.


learning_resources:

Use general resources only:

Allowed:

- Official documentation
- Online tutorials
- Practice platforms
- Open-source projects
- Hands-on projects


Do NOT include:

- Fake courses
- Fake certifications
- Fake books
- URLs



====================
RECOMMENDED PROJECTS RULES
====================

Generate projects ONLY if improvement areas exist.


Projects must:

- Directly improve missing skills.
- Match target job.
- Match candidate level.
- Be realistic portfolio projects.


Each project must contain:

project_name

related_skill

description

expected_outcome


Do NOT suggest:

- Research-level projects
- Enterprise-scale systems
- Senior engineering tasks



====================
CAREER ADVICE RULES
====================

- Keep it short (1-2 sentences).
- Focus only on improvement areas.
- Mention practical learning or portfolio actions.
- Do not mention promotions.
- Do not mention leadership unless supported.



====================
CANDIDATE PROFILE
====================

{json.dumps(candidate_profile, indent=4)}



====================
ATS REPORT
====================

{json.dumps(ats_report, indent=4)}



====================
SKILL GAP ANALYSIS
====================

{json.dumps(skill_gap_report, indent=4)}



====================
JOB PROFILE
====================

{json.dumps(job_profile, indent=4)}

"""

In [79]:
career_result = generate_json_text(
    career_prompt
)


print(career_result)

```json
{
    "target_role": "Machine Learning Engineer",
    "candidate_level": "Junior",
    "strengths": [
        "Experience with Python programming",
        "Knowledge of Machine Learning algorithms",
        "Hands-on experience with Deep Learning frameworks like TensorFlow and PyTorch",
        "Worked with Computer Vision and NLP",
        "Experience with deploying ML models using APIs or Streamlit",
        "Data preprocessing and model evaluation"
    ],
    "improvement_areas": [
        "sql databases"
    ],
    "learning_plan": [
        {
            "skill_name": "sql databases",
            "priority": "High",
            "reason": "Understanding and manipulating data from SQL databases is crucial for Machine Learning Engineers as it allows them to work with large datasets efficiently and extract meaningful insights.",
            "learning_resources": [
                "Basic SQL syntax such as SELECT, JOIN, WHERE clauses",
                "Online tutorials on SQL 

In [80]:
career_report = extract_json(
    career_result
)


print(
    json.dumps(
        career_report,
        indent=4
    )
)

{
    "target_role": "Machine Learning Engineer",
    "candidate_level": "Junior",
    "strengths": [
        "Experience with Python programming",
        "Knowledge of Machine Learning algorithms",
        "Hands-on experience with Deep Learning frameworks like TensorFlow and PyTorch",
        "Worked with Computer Vision and NLP",
        "Experience with deploying ML models using APIs or Streamlit",
        "Data preprocessing and model evaluation"
    ],
    "improvement_areas": [
        "sql databases"
    ],
    "learning_plan": [
        {
            "skill_name": "sql databases",
            "priority": "High",
            "reason": "Understanding and manipulating data from SQL databases is crucial for Machine Learning Engineers as it allows them to work with large datasets efficiently and extract meaningful insights.",
            "learning_resources": [
                "Basic SQL syntax such as SELECT, JOIN, WHERE clauses",
                "Online tutorials on SQL database

In [81]:
with open(
    "outputs/career_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        career_report,
        f,
        indent=4,
        ensure_ascii=False
    )


print("All reports saved successfully")

All reports saved successfully


# Test

In [82]:
import json


with open(
    "outputs/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate_data = json.load(f)


with open(
    "outputs/job_profile.json",
    "r",
    encoding="utf-8"
) as f:
    job_data = json.load(f)


with open(
    "outputs/ats_report.json",
    "r",
    encoding="utf-8"
) as f:
    ats_data = json.load(f)


with open(
    "outputs/skill_gap_report.json",
    "r",
    encoding="utf-8"
) as f:
    skill_gap_data = json.load(f)


with open(
    "outputs/career_report.json",
    "r",
    encoding="utf-8"
) as f:
    career_data = json.load(f)


print("All files loaded successfully")

All files loaded successfully
